# 01 - Ingesta Bronze: RUES (Registro Mercantil)

Trabajo Práctico 1 - Sección 3 (Fuente 1): Pipeline de ingesta PySpark

## Fuente

* **Dataset**: Personas Naturales, Personas Jurídicas y Entidades Sin Ánimo de Lucro (RUES)
* **API**: `https://www.datos.gov.co/resource/c82u-588k.json` (Socrata / SoQL)
* **Volumen total en la fuente**: 9.407.309 registros (verificado con `$select=count(*)`)
* **Destino**: `Datos_Empresas.bronze.DE_Semiestructurado_RegistroMercantil_Api`

## Estrategia de carga: dos modos con el flag `MODO_DIARIO`

Este pipeline soporta dos formas de cargar RUES, controladas por una sola variable booleana:

* **`MODO_DIARIO = False`** (carga por mes): se cambian las variables `ANIO` y `MES` y se filtra por `fecha_actualizacion` para traer ese mes completo. La escritura usa `replaceWhere`: si el mes ya se había cargado antes, se trunca (borra) solo ese rango y se reemplaza con lo que traiga la API en esta ejecución; los demás meses ya cargados no se tocan.
* **`MODO_DIARIO = True`** (carga diaria): se ignoran `ANIO`/`MES` y se filtra `fecha_actualizacion` para traer solo **hoy y ayer**. La escritura usa `mode("append")`, pero antes se excluyen (por `left_anti` join) las filas que ya existan en la tabla con la misma llave de negocio (`codigo_camara`, `matricula`) y la misma `fecha_actualizacion`, así que volver a correrlo el mismo día no duplica nada — solo se agregan los registros realmente nuevos.

En ambos casos el histórico ya cargado (meses u otros días anteriores) nunca se reprocesa ni se duplica.

## Regla de inmutabilidad (Capa Bronze)

* No se renombran columnas: se conservan exactamente los nombres que entrega la API (`codigo_camara`, `razon_social`, etc.).
* No se hace *casting* destructivo: todas las columnas de RUES son de tipo `text` en el origen, así que se dejan como texto (no se convierten fechas a `DATE` ni números a `INT`/`DOUBLE`). Esa limpieza es tarea de la futura Capa Silver.
* Solo se agregan las columnas de auditoría obligatorias: `_ingested_at` y `_source`.

In [ ]:
%python
import requests
import pandas as pd
from datetime import datetime, timedelta
from pyspark.sql import functions as F

BASE_URL = "https://www.datos.gov.co/resource/c82u-588k.json"
LIMIT = 50000  # tamaño de página soportado por Socrata

# --- Flag principal: True = carga diaria (hoy y ayer, append sin duplicar) ---
# ---              False = carga por mes (ANIO/MES, con replaceWhere)     ---
MODO_DIARIO = False

# Solo se usan si MODO_DIARIO = False
ANIO = 2026
MES = 8
# ------------------------------------------------------


def rango_mes(anio, mes):
    inicio = datetime(anio, mes, 1)
    fin = datetime(anio + 1, 1, 1) if mes == 12 else datetime(anio, mes + 1, 1)
    return inicio.strftime("%Y/%m/%d"), fin.strftime("%Y/%m/%d")


def rango_diario():
    hoy = datetime.now().date()
    ayer = hoy - timedelta(days=1)
    manana = hoy + timedelta(days=1)  # límite superior exclusivo, incluye todo el día de hoy
    return ayer.strftime("%Y/%m/%d"), manana.strftime("%Y/%m/%d")


# fecha_actualizacion llega como texto 'YYYY/MM/DD HH:MM:SS...', comparable como string
if MODO_DIARIO:
    FECHA_INICIO, FECHA_FIN = rango_diario()
    print(f"Modo: DIARIO (hoy y ayer) -> {FECHA_INICIO} a {FECHA_FIN}")
else:
    FECHA_INICIO, FECHA_FIN = rango_mes(ANIO, MES)
    print(f"Modo: MENSUAL -> {ANIO}-{MES:02d} ({FECHA_INICIO} a {FECHA_FIN})")

WHERE_CLAUSE = f"fecha_actualizacion >= '{FECHA_INICIO}' AND fecha_actualizacion < '{FECHA_FIN}'"

print(f"Fuente: {BASE_URL}")
print(f"Filtro aplicado: {WHERE_CLAUSE}")

---

## Paso 1: Contar registros disponibles con el filtro aplicado

In [ ]:
%python
def contar_registros(where=None):
    params = {"$select": "count(*)"}
    if where:
        params["$where"] = where
    resp = requests.get(BASE_URL, params=params)
    resp.raise_for_status()
    return int(resp.json()[0]["count"])


total_registros = contar_registros(WHERE_CLAUSE)
print(f"Total de registros de RUES para el rango {FECHA_INICIO} - {FECHA_FIN}: {total_registros:,}")

---

## Paso 2: Descargar los datos paginando (HTTP GET + `$limit`/`$offset`)

In [0]:
%python
MAX_REGISTROS = total_registros


def descargar_datos(max_registros, where=None):
    registros = []
    offset = 0

    while offset < max_registros:
        pagina_limit = min(LIMIT, max_registros - offset)
        params = {"$limit": pagina_limit, "$offset": offset}
        if where:
            params["$where"] = where
        resp = requests.get(BASE_URL, params=params)
        resp.raise_for_status()
        pagina = resp.json()

        if not pagina:
            break

        registros.extend(pagina)
        offset += LIMIT
        print(f"Descargados {len(registros)} de {max_registros} registros...")

    return registros


registros = descargar_datos(MAX_REGISTROS, WHERE_CLAUSE)
df_pandas = pd.DataFrame(registros)
print(f"\nDataset descargado: {df_pandas.shape[0]} filas x {df_pandas.shape[1]} columnas")
df_pandas.head()

---

## Paso 3: Convertir a Spark DataFrame (sin transformar tipos) y agregar columnas de auditoría

In [0]:
%python
# Se respeta el tipo con el que Socrata entrega cada campo (todo texto en este dataset).
# No se hace ningún .astype()/cast manual: eso sería un casteo destructivo, prohibido en Bronze.
df_spark = spark.createDataFrame(df_pandas)

df_bronze = (
    df_spark
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source", F.lit(BASE_URL))
)

df_bronze.printSchema()
display(df_bronze.limit(10))

---

## Paso 4: Persistir en Delta Lake

* **Modo mensual** (`MODO_DIARIO = False`): si la tabla no existe todavía, se crea con el primer mes. Si ya existe, se usa `replaceWhere` con el mismo filtro de fecha del mes actual: eso borra e inserta de nuevo *solo* ese rango, dejando intactos los meses cargados en ejecuciones anteriores.
* **Modo diario** (`MODO_DIARIO = True`): se descargan hoy y ayer, se descartan (por `left_anti` join sobre `codigo_camara` + `matricula` + `fecha_actualizacion`) las filas que ya estén en la tabla, y solo lo verdaderamente nuevo se agrega con `mode("append")`. Así, correr el notebook varias veces el mismo día no genera duplicados.

In [ ]:
%python
TABLA_DESTINO = "Datos_Empresas.bronze.DE_Semiestructurado_RegistroMercantil_Api"

tabla_existe = spark.catalog.tableExists(TABLA_DESTINO)

if MODO_DIARIO:
    LLAVE_NEGOCIO = ["codigo_camara", "matricula", "fecha_actualizacion"]

    if tabla_existe:
        existentes = (
            spark.table(TABLA_DESTINO)
            .filter(WHERE_CLAUSE)
            .select(*LLAVE_NEGOCIO)
            .distinct()
        )
        df_nuevos = df_bronze.join(existentes, on=LLAVE_NEGOCIO, how="left_anti")
    else:
        df_nuevos = df_bronze

    (
        df_nuevos.write
        .format("delta")
        .mode("append")
        .saveAsTable(TABLA_DESTINO)
    )

    print(
        f"Modo diario: {df_nuevos.count()} registros nuevos agregados "
        f"(de {df_bronze.count()} descargados en el rango {FECHA_INICIO} - {FECHA_FIN}), sin duplicar."
    )
else:
    writer = df_bronze.write.format("delta").mode("overwrite")

    if tabla_existe:
        # Trunca solo el rango de fecha_actualizacion del mes actual; no toca otros meses ya cargados
        writer = writer.option("replaceWhere", WHERE_CLAUSE)
        print(f"Tabla existente: se trunca y recarga solo el rango {FECHA_INICIO} - {FECHA_FIN}")
    else:
        writer = writer.option("overwriteSchema", "true")
        print("Tabla nueva: se crea con este primer mes cargado")

    writer.saveAsTable(TABLA_DESTINO)
    print(f"Tabla Delta actualizada: {TABLA_DESTINO} (mes {ANIO}-{MES:02d})")

In [0]:
%sql
-- Verificación rápida de la ingesta
DESCRIBE EXTENDED Datos_Empresas.bronze.DE_Semiestructurado_RegistroMercantil_Api;

In [0]:
%sql
SELECT COUNT(*) AS total_filas, MIN(_ingested_at) AS primera_carga, MAX(_ingested_at) AS ultima_carga
FROM Datos_Empresas.bronze.DE_Semiestructurado_RegistroMercantil_Api;


In [0]:
%sql
-- Ver cuántas filas hay cargadas por cada mes (fecha_actualizacion) para confirmar
-- que se van acumulando meses sin duplicar
SELECT LEFT(fecha_actualizacion, 7) AS mes, COUNT(*) AS filas
FROM Datos_Empresas.bronze.DE_Semiestructurado_RegistroMercantil_Api
GROUP BY LEFT(fecha_actualizacion, 7)
ORDER BY mes;

In [0]:
%sql
SELECT * FROM Datos_Empresas.bronze.DE_Semiestructurado_RegistroMercantil_Api LIMIT 10;

---

## Siguiente paso

Continuar con [`02_Ingesta_Bronze_TRM.ipynb`](02_Ingesta_Bronze_TRM.ipynb) para la fuente complementaria con carga incremental.